In [3]:
import pandas as pd
import numpy as np

In [27]:
def assign_paabd(dat,train_dat = None,assign_kinase = False,get_sub_15 = True):
    if get_sub_15:
        dat["substrate_seq_15aa"] = dat.apply(lambda s: s["substrate_seq"][s["p_ind"]-1-7:s["p_ind"]-1+8],axis = 1)
    dat = dat.loc[dat["substrate_seq_15aa"].apply(lambda s:len(s))==15,:]
    dat.loc[:,"SH2"] = dat["substrate_seq_15aa"].apply(lambda s: s[7]=='Y')
    dat.loc[:,"b14-3-3"] = dat["substrate_seq_15aa"].apply(lambda s: ((s[7]=='S')|(s[7]=='T'))&(s[4]=='R')&(s[9]=='P'))
    dat.loc[:,"WW"] = dat["substrate_seq_15aa"].apply(lambda s: ((s[7]=='S')|(s[7]=='T'))&(s[8] in ['P','L','V','I']))
    dat.loc[:,"FHA"] = dat["substrate_seq_15aa"].apply(lambda s: ((s[7]=='T'))&(s[10] in ["D"]))
    if assign_kinase:
        dat = pd.merge(dat,train_dat.loc[:,["kinase","kinase_seq"]].drop_duplicates(), on = "kinase_seq")
    return dat 

In [28]:
blib_kinases_with_acc = pd.read_csv('../screening_analysis/blib_kinases_with_acc.csv',header = None)
blib_kinases_with_acc.columns = ['kinase_type','kinase','kinase_seq','kinase_acc']
dat_in_screen = pd.read_pickle('./251203_ksi_reduced_df_tp.pkl')
dat_in_screen = pd.merge(blib_kinases_with_acc,dat_in_screen,left_on = 'kinase',right_on = 'kinase')

In [29]:
dat_in_screen.columns = ['kinase_type', 'kinase', 'kinase_seq', 'kinase_acc', 'substrate_seq_15aa',
       'positive', 'paabd_hit', 'paabd_all', 'n_paabd', 'hit_count',
       'binding_max', 'binding_best_hit', 'lfsr_min', 'lfsr_best_hit',
       'fp_sub', 'true_positive']

In [30]:
dat_in_screen = assign_paabd(dat_in_screen,get_sub_15 = False)

In [31]:
dat_in_screen = dat_in_screen.loc[~dat_in_screen['kinase'].isin(['KS6B1_HUMAN_active','PLK1_HUMAN_active']),:]

In [32]:
dat_in_screen_tp = dat_in_screen.loc[dat_in_screen['true_positive']]

In [33]:
dat_in_screen_tp_wpaabd = pd.concat([dat_in_screen_tp.loc[((dat_in_screen_tp['paabd_hit'].apply(lambda s: 'fha' in s))&dat_in_screen_tp['FHA']),:],
dat_in_screen_tp.loc[((dat_in_screen_tp['paabd_hit'].apply(lambda s: 'ww' in s))&dat_in_screen_tp['WW']),:],
dat_in_screen_tp.loc[((dat_in_screen_tp['paabd_hit'].apply(lambda s: '1433' in s))&dat_in_screen_tp['b14-3-3']),:],
dat_in_screen_tp.loc[((dat_in_screen_tp['paabd_hit'].apply(lambda s: 'sh2' in s))&dat_in_screen_tp['SH2']),:]]).drop_duplicates(['kinase','substrate_seq_15aa'])

In [34]:
dat_in_screen_tp_wpaabd.shape

(3023, 20)

In [35]:
dat_in_screen_tp.loc[dat_in_screen_tp['substrate_seq_15aa']=='PHEEPSNSLMAIECR'].sort_values('binding_max',ascending = False).head(10)

,kinase_type,kinase,kinase_seq,kinase_acc,substrate_seq_15aa,positive,paabd_hit,paabd_all,n_paabd,hit_count,binding_max,binding_best_hit,lfsr_min,lfsr_best_hit,fp_sub,true_positive,SH2,b14-3-3,WW,FHA
19115,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,PHEEPSNSLMAIECR,True,1433;fha;ww,1433;fha;ww,3,3,1.623423,1.623423,0.0,0.0,False,True,False,False,True,False
2728,ST,PLK1_HUMAN,YVRGRFLGKGGFAKCFEISDADTKEVFAGKIVPKSLLLKPHQREKM...,P53350,PHEEPSNSLMAIECR,True,1433;fha;ww,1433;fha;ww,3,3,0.778031,0.778031,0.0,0.0,False,True,False,False,True,False
5858,ST,TBK1_HUMAN,WLLSDILGQGATANVFRGRHKKTGDLFAIKVFNNISFLRPVDVQMR...,Q9UHD2,PHEEPSNSLMAIECR,True,1433;fha;ww,1433;fha;ww,3,3,0.535463,0.535463,0.0,0.0,False,True,False,False,True,False
15243,ST,KKCC1_HUMAN,YKLQSEIGKGAYGVVRLAYNESEDRHYAMKVLSKKKLLKQYGFPRR...,Q8N5S9,PHEEPSNSLMAIECR,True,fha,1433;fha;ww,3,1,0.450815,0.450815,0.0,0.0,False,True,False,False,True,False
9237,ST,KGP1_HUMAN,FNIIDTLGVGGFGRVELVQLKSEESKTFAMKILKKRHIVDTRQQEH...,Q13976,PHEEPSNSLMAIECR,True,fha,fha;ww,2,1,0.444110,0.444110,0.0,0.0,False,True,False,False,True,False


In [36]:
dat_in_screen_tp.loc[dat_in_screen_tp['kinase']=='IKKB_HUMAN'].sort_values('binding_max',ascending = False).head(10)

,kinase_type,kinase,kinase_seq,kinase_acc,substrate_seq_15aa,positive,paabd_hit,paabd_all,n_paabd,hit_count,binding_max,binding_best_hit,lfsr_min,lfsr_best_hit,fp_sub,true_positive,SH2,b14-3-3,WW,FHA
19115,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,PHEEPSNSLMAIECR,True,1433;fha;ww,1433;fha;ww,3,3,1.623423,1.623423,0.0,0.0,False,True,False,False,True,False
19512,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,VRKNHMYSCRFSRQC,True,1433;ww,1433;fha;ww,3,2,1.335851,1.335851,0.0,0.0,False,True,False,False,False,False
18975,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,LHRKIHVTPLIPGKS,True,1433,1433;fha;ww,3,1,1.119202,1.119202,0.0,0.0,False,True,False,False,True,False
18803,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,GQNGEEKSLPSANHP,True,fha;ww,fha;ww,2,2,1.106606,1.106606,0.0,0.0,False,True,False,False,True,False
18740,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,FLVRASESIPRAYAL,True,1433;fha;ww,1433;fha;ww,3,3,1.097164,1.097164,0.0,0.0,False,True,False,False,True,False
18539,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,APRPRVETSRAVPLA,True,ww,1433;fha;ww,3,1,1.021030,1.021030,0.0,0.0,False,True,False,False,False,False
18752,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,FSLLRSPSWEPFRDW,True,fha,fha;ww,2,1,0.912923,0.912923,0.0,0.0,False,True,False,False,False,False
19481,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,VIKDEALSDGDDLRD,True,1433;fha,1433;fha;ww,3,2,0.907860,0.907860,0.0,0.0,False,True,False,False,False,False
19014,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,LVKSVSESHTSCPAE,True,fha;ww,fha;ww,2,2,0.903012,0.903012,0.0,0.0,False,True,False,False,False,False
19098,ST,IKKB_HUMAN,WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCL...,O14920,PEGEGTESTVITGVD,True,fha,fha,1,1,0.896073,0.896073,0.0,0.0,False,True,False,False,False,False


In [37]:
dat_in_screen_tp_wpaabd.loc[(dat_in_screen_tp_wpaabd['kinase']=='TBK1_HUMAN')&(dat_in_screen_tp_wpaabd['substrate_seq_15aa']=='QGPSQTSTPTDVTAI'),:]

,kinase_type,kinase,kinase_seq,kinase_acc,substrate_seq_15aa,positive,paabd_hit,paabd_all,n_paabd,hit_count,binding_max,binding_best_hit,lfsr_min,lfsr_best_hit,fp_sub,true_positive,SH2,b14-3-3,WW,FHA
5913,ST,TBK1_HUMAN,WLLSDILGQGATANVFRGRHKKTGDLFAIKVFNNISFLRPVDVQMR...,Q9UHD2,QGPSQTSTPTDVTAI,True,fha,fha,1,1,1.241288,1.241288,0.0,0.0,False,True,False,False,True,True


In [38]:
dat = pd.read_pickle('./251130_ksi_inference_scores.pkl')

In [39]:
dat = pd.merge(dat,dat_in_screen_tp_wpaabd,left_on = ['kinase','substrate','mean'],right_on = ['kinase','substrate_seq_15aa','binding_max'])

In [40]:
import pandas as pd

def pick_dominant_hit(dat_in_screen: pd.DataFrame,
                      group_col: str = "substrate_seq_15aa",
                      score_col: str = "binding_max",
                      min_gap: float = 0.4) -> pd.DataFrame:
    """
    For each substrate, keep the row with the largest binding_max,
    but only if (top - second) > min_gap. If a group has <2 rows, it won't pass.
    """
    df = dat_in_screen.copy()

    # rank within each substrate by binding_max descending
    df["_rank"] = df.groupby(group_col)[score_col].rank(method="first", ascending=False)

    top = df[df["_rank"] == 1].copy()
    second = df[df["_rank"] == 2][[group_col, score_col]].rename(columns={score_col: "_second"})
    top = top.merge(second, on=group_col, how="left")

    top["_gap"] = top[score_col] - top["_second"]

    out = top[top["_gap"] > min_gap].drop(columns=["_rank", "_second", "_gap"])
    return out

# usage:
dominant_rows = pick_dominant_hit(dat)


## Number of highly specific substrates to repsective kinases in screening with PAABD

In [41]:
dominant_rows.groupby('kinase').count().iloc[:,:1]

,substrate
kinase,
ABL2_HUMAN,8
FRK_HUMAN,6
FYN_HUMAN,26
IKKB_HUMAN,1
JAK2_HUMAN_1,2
KGP1_HUMAN,2
KKCC1_HUMAN,9
KS6B1_HUMAN_wt,28
KSYK_HUMAN,14


In [42]:
dominant_rows.loc[(dominant_rows['kinase']=='IKKB_HUMAN'),:]

,substrate,kinase,binding_strength,kinase_global,paabd,mean,sd,lfsr,pre_1,mtx_1,...,binding_max,binding_best_hit,lfsr_min,lfsr_best_hit,fp_sub,true_positive,SH2,b14-3-3,WW,FHA
1263,PHEEPSNSLMAIECR,IKKB_HUMAN,1.623423,-1.153545,1433,1.623423,0.111486,0.0,413,1522,...,1.623423,1.623423,0.0,0.0,False,True,False,False,True,False


In [46]:
dominant_rows.loc[(dominant_rows['kinase']=='TBK1_HUMAN'),:].sort_values('binding_max').groupby('paabd').head(1)

,substrate,kinase,binding_strength,kinase_global,paabd,mean,sd,lfsr,pre_1,mtx_1,...,binding_max,binding_best_hit,lfsr_min,lfsr_best_hit,fp_sub,true_positive,SH2,b14-3-3,WW,FHA
1248,LYLLDRFSPFGRFKV,TBK1_HUMAN,0.696392,0.350864,1433,0.696392,0.116285,0.0,105,283,...,0.696392,0.696392,0.0,0.0,False,True,False,False,True,False
1354,EVETPNSTPPRRVQT,TBK1_HUMAN,0.985641,0.202587,ww,0.985641,0.114681,0.0,469,724,...,0.985641,0.985641,0.0,0.0,False,True,False,False,True,False
1112,QGPSQTSTPTDVTAI,TBK1_HUMAN,1.241288,0.945265,fha,1.241288,0.113755,0.0,76,960,...,1.241288,1.241288,0.0,0.0,False,True,False,False,True,True


### we picked the novel FHA1-compatible substrate and proceed from there
Specifically for TBK1 and IKKb, we picked the one with the highest binding_max energy

In [21]:
dominant_rows

,substrate,kinase,binding_strength,kinase_global,paabd,mean,sd,lfsr,pre_1,mtx_1,...,binding_max,binding_best_hit,lfsr_min,lfsr_best_hit,fp_sub,true_positive,SH2,b14-3-3,WW,FHA
13,AEDQEPTYCNMGHLS,FYN_HUMAN,0.989938,-0.568580,sh2,0.989938,0.113133,0.0,515,964,...,0.989938,0.989938,0.0,0.0,False,True,True,False,False,False
33,AKPDSSFYKGLDLNG,FRK_HUMAN,0.644943,-0.993242,sh2,0.644943,0.112899,0.0,3397,1192,...,0.644943,0.644943,0.0,0.0,False,True,True,False,False,False
81,CDLLRQQYEEEQEGK,ABL2_HUMAN,0.940614,-1.007143,sh2,0.940614,0.118938,0.0,103,73,...,0.940614,0.940614,0.0,0.0,False,True,True,False,False,False
98,DASSQDCYDIPRAFP,KSYK_HUMAN,0.951486,-0.794144,sh2,0.951486,0.111921,0.0,1169,1013,...,0.951486,0.951486,0.0,0.0,False,True,True,False,False,False
110,DEEVDEMYREAPIDK,FYN_HUMAN,1.001913,-0.568580,sh2,1.001913,0.112532,0.0,613,568,...,1.001913,1.001913,0.0,0.0,False,True,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1337,TPVATSPTATDSTSG,TBK1_HUMAN,1.110186,0.202587,ww,1.110186,0.113519,0.0,653,1502,...,1.110186,1.110186,0.0,0.0,False,True,False,False,False,True
1338,TPVVSVTTPSLPPQG,KS6B1_HUMAN_wt,1.099469,0.452510,ww,1.099469,0.113443,0.0,179,374,...,1.099469,1.099469,0.0,0.0,False,True,False,False,True,False
1339,TQGHPDGTPPKLDTA,PLK1_HUMAN,1.298997,0.034903,ww,1.298997,0.116174,0.0,1142,2248,...,1.298997,1.298997,0.0,0.0,False,True,False,False,True,False
1340,VLEGEINTEDDDDED,KS6B1_HUMAN_wt,1.470530,0.452510,ww,1.470530,0.112221,0.0,273,913,...,1.470530,1.470530,0.0,0.0,False,True,False,False,False,True


In [20]:
dominant_rows.to_pickle('./kinase_specific_substrate_candidates.pkl')